# Run the pretrained CSI GMM across collection uploads

The firmware captures at a requested 100 Hz (the original rate, restored) and sends small full batches while capture continues. If RAM fills, capture pauses until an upload completes. Uploads and model windows are independent: this notebook buffers packets across batch boundaries to form configurable analysis windows. The default local profile uses the selected artifact's training settings. The short_1s profile remains available for exploratory comparison.

The frozen GMM is unchanged. Its states are unsupervised clusters, not occupancy labels or person counts. Changing sampling conditions can change feature distributions; predictions here do not establish accuracy at 50 Hz. Lambda and AWS resources are unchanged.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

INPUT_CSV = Path(os.environ.get('CSI_COLLECTION_INPUT_CSV', r'E:\Downloads\results (1).csv'))
MODEL_FILENAME = 'standalone_gmm.py'
SAVE_RESULTS = True
candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([parent, parent/'modeling_and_analysis', parent/'drafts/modeling_and_analysis'])
WORKSPACE = next((p.resolve() for p in candidates if (p/MODEL_FILENAME).is_file()), None)
if WORKSPACE is None:
    raise FileNotFoundError('Run beside standalone_gmm.py or from the repository root.')
if not INPUT_CSV.is_file():
    raise FileNotFoundError(f'Update INPUT_CSV: {INPUT_CSV}')
sys.path.insert(0, str(WORKSPACE))
from collection_pipeline import load_collection_export, predict_collection_batches
MODEL_PATH = WORKSPACE / MODEL_FILENAME
spec = importlib.util.spec_from_file_location('collection_frozen_gmm', MODEL_PATH)
model = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model)
# All four settings are editable; training metadata supplies a comparison only.
WINDOW_PROFILES = {
    'bundle_training': dict(model.PROVENANCE['metadata']['fast_parameters']),
    'short_1s': dict(window_seconds=1.0, hop_seconds=0.2, min_packets=12, max_gap_seconds=0.25),
}
ACTIVE_PROFILE = 'bundle_training'
PARAMETERS = dict(WINDOW_PROFILES[ACTIVE_PROFILE])
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_ROOT = Path(os.environ.get('CSI_INFERENCE_OUTPUT_DIR', str(WORKSPACE/'reports'/'collection_inference')))
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})
print('Python:', sys.executable)
print('Input:', INPUT_CSV)
print('Frozen model:', MODEL_PATH)
print('Window configuration:', PARAMETERS)
print('Output:', OUTPUT_DIR if SAVE_RESULTS else 'exports disabled')


## Decode and assemble the uploads

The shared local pipeline accepts DynamoDB `M`, `L`, `N`, `S`, and `BOOL` wrappers or plain sample arrays. Device microsecond timestamps remain exact integers; they are never truncated to 32 bits or converted to UTC for inference. `timestamp_received` is retained only as a network diagnostic, not as the acquisition clock.

Rows may be shuffled. Batches are ordered by numeric batch ID and packets by sequence. Identical repeated packets are removed. Conflicting duplicates and clock regressions reject the affected capture. Device, boot session, AP, channel, bandwidth, and CSI length define capture identity. The expected CSI length is 128 signed I/Q values.


In [ ]:
batches, load_issues = load_collection_export(INPUT_CSV)
result = predict_collection_batches(batches, model, parameters=PARAMETERS)
profile_rows = []
for profile_name, profile_parameters in WINDOW_PROFILES.items():
    candidate = result if profile_name == ACTIVE_PROFILE else predict_collection_batches(batches, model, profile_parameters)
    profile_rows.append(dict(
        profile=profile_name, **candidate['window_parameters'],
        complete_windows=len(candidate['windows']),
        accepted_windows=sum(row['status'] == 'ok' for row in candidate['windows']),
        settings_match_training=candidate['settings_match_training']))
profile_comparison = pd.DataFrame(profile_rows)
display(profile_comparison)
if not result['settings_match_training']:
    display(Markdown('**Exploratory window settings:** the GMM coefficients are unchanged. Accepted coverage and cluster assignments do not validate accuracy for this observation duration.'))
input_issues = pd.DataFrame(load_issues + result['issues'], columns=['csv_row', 'sample_index', 'reason'])
packet_table = pd.DataFrame(result['packets'])
capture_summary = pd.DataFrame(result['captures'])
batch_summary = pd.DataFrame(result['batches'])
if not batch_summary.empty:
    batch_summary['within_batch_rate_hz'] = np.where(
        batch_summary.duration_s.gt(0),
        (batch_summary.packet_count-1)/batch_summary.duration_s.replace(0, np.nan), 0.0)
window_predictions = pd.DataFrame(result['windows'])
if window_predictions.empty:
    window_predictions = pd.DataFrame(columns=[
        'capture_id', 'window_center_s', 'window_start_s', 'packet_count', 'max_gap_s',
        'observed_max_gap_s', 'quality_ok', 'status', 'state', 'rejection_reason'])
accepted_windows = window_predictions[window_predictions.status.eq('ok')].copy()
print(f'{len(batches)} decoded export rows; {len(packet_table)} unique indexed packets; {len(input_issues)} input issues')
if not packet_table.empty:
    display(packet_table.drop(columns=['csi_raw']).head())
if not batch_summary.empty:
    display(batch_summary.head(10))
if not input_issues.empty:
    display(input_issues)


## Compare and tune window settings

Edit `WINDOW_PROFILES` and `ACTIVE_PROFILE` in the setup cell. The selected local default is the active model's training profile. The editable short_1s comparison uses a **1-second window**, **0.2-second hop**, **at least 12 packets**, and **maximum 0.25-second gap** including edges. At 100 Hz this normally provides about 100 packets per window; it also permits some short, well-covered observations in the older export. These are starting parameters, not a validated optimum.

The original training settings remain a comparison profile, not a runtime restriction. The table reports coverage for both profiles; their overlapping window counts differ because the hop differs. Acceptance is a sampling check, not a model accuracy metric. The supplied older GMM was fitted on 5-second features; the comparison table shows the training settings of whichever inference artifact is selected. outputs under changed parameters carry `inference_mode=exploratory_window_override` and need validation or retraining before interpretation as reliable states.

The buffer retains pending packets across uploads and emits complete windows once. Long acquisition gaps stay unknown under the selected gap limit. Trailing windows are not padded. Missing sequences cannot distinguish discarded uploads from other losses; serial logs provide firmware queue diagnostics.


In [ ]:
print(f'Complete windows: {len(window_predictions):,}; accepted: {len(accepted_windows):,}; '
      f'unknown: {len(window_predictions)-len(accepted_windows):,}')
display(capture_summary)
if not window_predictions.empty:
    display(window_predictions.groupby(['capture_id', 'rejection_reason']).size().rename('windows').reset_index())
    display(window_predictions[['capture_id', 'window_start_s', 'packet_count',
                                'observed_max_gap_s', 'source_batch_ids', 'quality_ok',
                                'state', 'inference_mode', 'rejection_reason']].head(12))
if accepted_windows.empty:
    display(Markdown('**No valid state predictions for this export.** Inspect coverage and rejection reasons; buffering cannot recover unobserved acquisition time.'))
else:
    display(accepted_windows.groupby(['capture_id', 'state']).size().rename('accepted_windows').reset_index())


## Inspect coverage, upload boundaries, and predicted states

All axes use elapsed acquisition time from the first packet of that capture. Grey ticks show packets; faint blue lines show the first packet in each uploaded batch. Red windows are unknown and green windows passed the sampling checks.

`observed_max_gap_s` is measured directly, including edges even for sparse or empty windows. The frozen feature extractor's `max_gap_s` retains its conservative default for windows with fewer than the selected minimum packets. Neither upload duration nor receipt jitter is inserted into the acquisition timeline.


In [ ]:
coverage_figure = None
plotted_ids = window_predictions.capture_id.dropna().unique()
if len(plotted_ids):
    coverage_figure, axes = plt.subplots(len(plotted_ids), 3, figsize=(16, 3*len(plotted_ids)), squeeze=False)
    for row_index, capture_id in enumerate(plotted_ids):
        windows = window_predictions[window_predictions.capture_id.eq(capture_id)]
        arrivals = packet_table[packet_table.capture_id.eq(capture_id)]
        left, middle, right = axes[row_index]
        left.plot(arrivals.time_s, np.zeros(len(arrivals)), '|', color='#475569', markersize=12)
        for boundary in arrivals.groupby('batch_id', sort=False).time_s.min():
            left.axvline(boundary, color='#2563eb', alpha=0.12, linewidth=0.6)
        left.set(title=f'{capture_id}: arrivals / upload batches', xlabel='Acquisition time (s)', yticks=[])
        for good, color, label in [(False, '#dc2626', 'unknown'), (True, '#16a34a', 'accepted')]:
            subset = windows[windows.quality_ok.eq(good)]
            middle.scatter(subset.window_center_s, subset.packet_count, s=9, color=color, label=label)
        middle.axhline(PARAMETERS['min_packets'], color='#334155', linestyle='--', linewidth=1)
        middle.set(title='Packets per model window', xlabel='Window center (s)', ylabel='Packets')
        middle.legend(loc='upper right')
        right.plot(windows.window_center_s, windows.observed_max_gap_s, color='#dc2626', linewidth=1)
        right.axhline(PARAMETERS['max_gap_seconds'], color='#334155', linestyle='--', linewidth=1)
        right.set(title='Observed gap including window edges', xlabel='Window center (s)', ylabel='Gap (s)')
    coverage_figure.tight_layout()
    plt.show()
else:
    print('No complete windows. Inspect short/unsupported captures and trailing packets above.')

state_figure = None
if not accepted_windows.empty:
    ids = accepted_windows.capture_id.unique()
    state_figure, axes = plt.subplots(len(ids), 2, figsize=(14, 3*len(ids)), squeeze=False)
    for row_index, capture_id in enumerate(ids):
        current = accepted_windows[accepted_windows.capture_id.eq(capture_id)]
        axes[row_index, 0].scatter(current.window_center_s, current.state.astype(int), s=12)
        axes[row_index, 0].set(title=f'{capture_id}: GMM cluster', xlabel='Window center (s)', ylabel='Cluster ID',
                               yticks=range(len(model.COEFFICIENTS['model']['weights_'])))
        for column in current.filter(regex=r'^membership_').columns:
            axes[row_index, 1].scatter(current.window_center_s, current[column], s=10, label=column)
        axes[row_index, 1].set(title='Component membership', xlabel='Window center (s)', ylabel='Membership', ylim=(-0.05, 1.05))
        axes[row_index, 1].legend()
    state_figure.tight_layout()
    plt.show()


## Export predictions and provenance

Every run writes separate reports. Historical exports with long gaps can still have zero accepted windows; continuous exports may produce predictions across many small uploads. No fixed acceptance count is assumed and no model is fitted here. Lambda is not used or changed.


In [ ]:
run_summary = {
    'run_id_utc': RUN_ID, 'input_csv': str(INPUT_CSV.resolve()),
    'input_sha256': hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest(),
    'model_path': str(MODEL_PATH), 'model_sha256': hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    'pipeline_sha256': hashlib.sha256((WORKSPACE/'collection_pipeline.py').read_bytes()).hexdigest(),
    'model_provenance': model.PROVENANCE, 'window_parameters': result['window_parameters'],
    'active_profile': ACTIVE_PROFILE, 'training_window_parameters': result['training_window_parameters'],
    'settings_match_training': result['settings_match_training'],
    'profile_comparison': profile_comparison.to_dict('records'),
    'expected_csi_length': 128, 'fitted_during_this_run': False,
    'decoded_export_rows': len(batches), 'packet_records': len(packet_table),
    'input_issues': len(input_issues), 'captures': len(capture_summary),
    'complete_windows': len(window_predictions), 'accepted_windows': len(accepted_windows),
    'unknown_windows': len(window_predictions)-len(accepted_windows),
    'state_counts': {str(int(state)): int(count) for state, count in accepted_windows.state.value_counts().items()},
    'interpretation': 'Unsupervised clusters only; insufficient coverage remains unknown; no occupancy accuracy calculated.',
}
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    capture_summary.to_csv(OUTPUT_DIR/'capture_summary.csv', index=False)
    profile_comparison.to_csv(OUTPUT_DIR/'profile_comparison.csv', index=False)
    batch_summary.to_csv(OUTPUT_DIR/'batch_summary.csv', index=False)
    window_predictions.to_csv(OUTPUT_DIR/'window_predictions.csv', index=False)
    input_issues.to_csv(OUTPUT_DIR/'input_issues.csv', index=False)
    packet_table.drop(columns=['csi_raw'], errors='ignore').to_csv(OUTPUT_DIR/'packet_index.csv', index=False)
    if coverage_figure is not None:
        coverage_figure.savefig(OUTPUT_DIR/'sampling_coverage.png', dpi=150, bbox_inches='tight')
    if state_figure is not None:
        state_figure.savefig(OUTPUT_DIR/'state_membership.png', dpi=150, bbox_inches='tight')
    (OUTPUT_DIR/'run_summary.json').write_text(json.dumps(run_summary, indent=2, allow_nan=False), encoding='utf-8')
    print('Saved results:', OUTPUT_DIR)
print(json.dumps({key: value for key, value in run_summary.items() if key not in
                 {'model_provenance', 'input_sha256', 'model_sha256', 'pipeline_sha256'}}, indent=2))


## Write a new inference model from selected analysis results

After fitting and comparing models with the chosen window settings in a modeling notebook, save the selected fitted pipeline with its feature order and training window settings in a `.joblib` bundle. This inference notebook does not fit a new model from its cluster predictions.

Set `SELECTED_BUNDLE` below to that analysis result to write a new standalone inference module. The exporter includes its fitted preprocessing, GMM coefficients and saved window settings, and checks agreement with sklearn before replacement. It does not automatically select the old experiment. After export, set `MODEL_FILENAME = 'models/selected_gmm.py'` in setup and rerun to evaluate the new model. Lambda is unchanged.


In [ ]:
# Choose a fitted analysis result explicitly; None leaves existing artifacts intact.
SELECTED_BUNDLE = None  # e.g. WORKSPACE/'models'/'new_run'/'selected.joblib'
FEATURE_RESULTS = None  # optional feature CSV path or DataFrame from that run
EXPORT_PATH = WORKSPACE/'models'/'selected_gmm.py'
if SELECTED_BUNDLE is not None:
    from export_gmm import export_gmm
    export_report = export_gmm(SELECTED_BUNDLE, EXPORT_PATH, feature_frame=FEATURE_RESULTS)
    print(json.dumps(export_report, indent=2))
else:
    print('Select a fitted analysis bundle above to export a new inference model.')
